# First Attempt

In [55]:
import torch
import numpy as np

## Load/ Preprocess Data

In [56]:
import pandas as pd
import numpy as np
import wfdb
import ast

#provided function to load data with information in the agg_df (modified to return a numpy array)
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data]).astype(np.float32)
    return data

# path to dta and selected sampling rate
path = "physionet.org/files/ptb-xl/1.0.3/"
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x)) #changes class distribution to dict

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1] #only take diagnostic scps

classes = agg_df.index.tolist()

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel(class_list):
    return np.array([1 if cls in class_list else 0 for cls in classes])

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(key)
    return list(set(tmp))

# Apply diagnostic superclass and encoding multilabel
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)




In [57]:
agg_df.index

Index(['NDT', 'NST_', 'DIG', 'LNGQT', 'NORM', 'IMI', 'ASMI', 'LVH', 'LAFB',
       'ISC_', 'IRBBB', '1AVB', 'IVCD', 'ISCAL', 'CRBBB', 'CLBBB', 'ILMI',
       'LAO/LAE', 'AMI', 'ALMI', 'ISCIN', 'INJAS', 'LMI', 'ISCIL', 'LPFB',
       'ISCAS', 'INJAL', 'ISCLA', 'RVH', 'ANEUR', 'RAO/RAE', 'EL', 'WPW',
       'ILBBB', 'IPLMI', 'ISCAN', 'IPMI', 'SEHYP', 'INJIN', 'INJLA', 'PMI',
       '3AVB', 'INJIL', '2AVB'],
      dtype='str')

In [58]:
# Further example code 
"""
Y_s = Y.head(10).copy()

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y_s.strat_fold != test_fold)]
y_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y_s.strat_fold == test_fold)]
y_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass"""

'\nY_s = Y.head(10).copy()\n\n# Load raw signal data\nX = load_raw_data(Y_s, sampling_rate, path)\n\n# Split data into train and test\ntest_fold = 10\n# Train\nX_train = X[np.where(Y_s.strat_fold != test_fold)]\ny_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass\n# Test\nX_test = X[np.where(Y_s.strat_fold == test_fold)]\ny_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass'

In [59]:
#Y_s.diagnostic_superclass.to_numpy() #example line run of multilabel distribution
#Y['scp_codes']

In [60]:
Y['scp_codes']

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: object

## Data Prep

In [61]:
from sklearn.model_selection import train_test_split

Y_s = Y.sample(2100)

Y_transformed = np.stack(Y_s.diagnostic_superclass.values).astype(np.float32)

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(X, Y_transformed, test_size=0.1, random_state=56)

In [62]:
X_train_torch = torch.tensor(X_train) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

In [63]:
train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


## CNN

In [64]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


class CNet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    #starting with 96

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.fc1 = nn.Linear(512 * 125, 1000)
    self.fc2 = nn.Linear(1000, 500)
    self.fc3 = nn.Linear(500, 44)

  def forward(self, x):
    #x = self.pool(F.relu(self.conv12(F.relu(self.conv11(x)))))
    #x = self.pool(F.relu(self.conv22(F.relu(self.conv21(x)))))
    #x = self.pool(F.relu(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
    #x = F.relu(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x))))))

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [77]:
def m_train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()
    #loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([5]).to(device))

    # MASKED LOSS 
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([5]).to(device), reduction="none").to(device)

    lr = 1e-3
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            mask = torch.ones(y.shape[1], device=device)
            mask[4] = 0.1
            masked_loss = (loss*mask.mean())
            #running_loss += loss.item()

            final_loss = masked_loss.sum() / (mask.sum() * loss.shape[0])
            running_loss += final_loss.item()
            final_loss.backward()
            #loss.backward() #compute gradients

            opt.step() #apply gradients
            #scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")

def train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([5]).to(device))

    lr = 1e-3
    opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    #scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
            #scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")


In [66]:
X.shape

(2100, 12, 1000)

In [78]:
def jaccard_score(y_pred, y_true):
    intersection = ((y_pred == 1) & (y_true == 1)).sum()
    union = ((y_pred == 1) | (y_true == 1)).sum()

    if union == 0:
        return 0.0
    
    return intersection / union

def precision(y_pred, y_true):
    f_p = ((y_pred == 1) & (y_true == 1)).sum()
    t_p = ((y_pred == 1) & (y_true == 0)).sum()
    if t_p + f_p == 0:
        return 0.0
    return t_p / (t_p + f_p)

def recall(y_pred, y_true):
    t_p = ((y_pred == 1) & (y_true == 1)).sum()
    f_n = ((y_pred == 0) & (y_true == 1)).sum()

    if t_p + f_n == 0:
        return 0.0
    return t_p / (t_p + f_n)

def f1(y_pred, y_true):
    m_p = precision(y_pred, y_true)
    m_r = recall(y_pred, y_true)
    if m_p + m_r == 0:
        return 0.0
    return 2 * (m_p*m_r) / (m_p+m_r)

def macro_precision(y_pred, y_true):
    y_pred = y_pred.int()
    y_true = y_true.int()
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_p = ((y_pred == 1) & (y_true == 0)).sum(dim=0)
    return t_p.float() / (t_p + f_p).float().clamp(min=1e-8)

def macro_recall(y_pred, y_true):
    y_pred = y_pred.int()
    y_true = y_true.int()
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_n = ((y_pred == 0) & (y_true == 1)).sum(dim=0)
    return t_p.float() / (t_p + f_n).float().clamp(min=1e-8)

def macro_f1(y_pred, y_true):
    m_p = macro_precision(y_pred, y_true)
    m_r = recall(y_pred, y_true)
    return 2 * m_p * m_r / (m_p + m_r).clamp(min=1e-8)


In [68]:
device = 'cuda'

In [69]:
#x, y = next(iter(train_dataloader))
#y_p = net(x.to(device))
#y_d = y.to(device)
#recall(y_p, y_d), precision(y_p, y_d), jaccard_score(y_p, y_d), f1(y_p, y_d)

In [79]:
net = CNet().to(device)
train_model(net, train_dataloader, test_dataloader, 30, 1e-2)

Epoch: 0 Batch 0
Epoch: 0 Batch 1
Epoch: 0 Batch 2
Epoch: 0 Batch 3
Epoch: 0 Batch 4
Epoch: 0 Batch 5
Epoch: 0 Batch 6
Epoch: 0 Batch 7
Epoch: 0 Batch 8
Epoch: 0 Batch 9
Epoch: 0 Batch 10
Epoch: 0 Batch 11
Epoch: 0 Batch 12
Epoch: 0 Batch 13
Epoch: 0 Batch 14
Epoch: 0 Batch 15
Epoch: 0 Batch 16
Epoch: 0 Batch 17
Epoch: 0 Batch 18
Epoch: 0 Batch 19
Epoch: 0 Batch 20
Epoch: 0 Batch 21
Epoch: 0 Batch 22
Epoch: 0 Batch 23
Epoch: 0 Batch 24
Epoch: 0 Batch 25
Epoch: 0 Batch 26
Epoch: 0 Batch 27
Epoch: 0 Batch 28
Epoch: 0 Batch 29
Epoch: 0 Batch 30
Epoch: 0 Batch 31
Epoch: 0 Batch 32
Epoch: 0 Batch 33
Epoch: 0 Batch 34
Epoch: 0 Batch 35
Epoch: 0 Batch 36
Epoch: 0 Batch 37
Epoch: 0 Batch 38
Epoch: 0 Batch 39
Epoch: 0 Batch 40
Epoch: 0 Batch 41
Epoch: 0 Batch 42
Epoch: 0 Batch 43
Epoch: 0 Batch 44
Epoch: 0 Batch 45
Epoch: 0 Batch 46
Epoch: 0 Batch 47
Epoch: 0 Batch 48
Epoch: 0 Batch 49
Epoch: 0 Batch 50
Epoch: 0 Batch 51
Epoch: 0 Batch 52
Epoch: 0 Batch 53
Epoch: 0 Batch 54
Epoch: 0 Batch 55
Ep

In [ ]:
for x_test, y_test in test_dataloader:
    val = (torch.sigmoid(net(x_test.to(device))) > 0.5).int()
    y_test = y_test.to(device).int()
    j_score = jaccard_score(val, y_test)
    f1_score = f1(val, y_test)
    p_score = precision(val, y_test)
    r_score = recall(val, y_test)
    m_f1_score = macro_f1(val, y_test)
    m_p_score = macro_precision(val, y_test)
    m_r_score = macro_recall(val, y_test)
    #print(j_score, f1_score, p_score, r_score)
    #print(m_f1_score)
    #print(m_p_score)
    #print(m_r_score)
    if j_score > 0.4:
        print(val, y_test)

tensor([0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.2000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       device='cuda:0')
tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')
tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')
tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,